## inference for mean differences 
2개 그룹의 평균이 동일한가?

In [71]:
# # _rd_00.py 
# 평균차 추론. 평균차 분산, 표준오차, 신뢰구간, 가설검정  
#   등분산 전제, 이분산 전제. 오류 정정(자유도 관련 계산)

import os
import numpy as np                          # numpy 라이브러리 전체. 
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네. 
import scipy as sci 
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis 
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp, ttest_ind, f


## 등분산 검정과 평균차 추론

등분산 가설이 기각되면, 이분산을 전제로 평균차에 대한 추론 진행. 

등분산 가설이 기각되지 않으면, 등분산을 전제로 추론 진행.

In [72]:
# 등분산 검정 시 적용 가능한 함수 
def variance_ratio_test(x, y, alpha) : # alpha=0.05):
    x = np.asarray(x)  # 이게 버릇이군. 이하 ndarray로 변환/유지한다는 뜻. 일관성 유지.
    y = np.asarray(y)

    n1 = len(x)
    n2 = len(y)

    s1 = np.var(x, ddof=1)   # 자유도를 일상적으로 지정하자. 디폴트로 넘어가다가 미세 차이 발생 가능.
    s2 = np.var(y, ddof=1)

    F_ratio = s1 / s2
    df1 = n1 - 1
    df2 = n2 - 1
                             # 분포 호출 시, cdf, sf, ppf, isf 참조. 1-cdf보다는 sf. 
    p_value = 2 * min( f.cdf(F_ratio, df1, df2),
                 f.sf(F_ratio, df1, df2) ) 
              #  1 - f.cdf(F, df1, df2))  # 이것보다 f.sf를 쓰라는, 그게 안정적이라는.

    c_interval = (
        F_ratio / f.ppf(1 - alpha/2, df1, df2),
        F_ratio / f.ppf(alpha/2, df1, df2)
        )

    return F_ratio, p_value, c_interval


In [73]:

# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat_r = pd.read_csv(dat_url) 
df_dat_r.head() 


,i,gender,ht
0,1,1,159.9
1,2,2,157.5
2,3,2,158.0
3,4,2,154.2
4,5,1,163.3


In [74]:
# df_dat에서 변수 생성 등 
# gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
# hgt = df_dat['ht'].to_numpy()
# gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환. 
# df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

# hgt_m = hgt[gndr == 1]  # 성별 1, m 그룹 키. 그룹별 자료 분리, 배열은 hgt 하나임.
# hgt_f = hgt[gndr == 0]  #     0, f. 키 그룹별 자료 분리

# 데이터 너무 많음. 일부만 선택해서 연습. 
# 추출 비율 약 2% 정도? 
# 단순추출, 층화추출. 


In [75]:
# # 1. 연습용 자료, 단순 무작위 추출 (2%, 대략 400개)
n_frac = 0.02 
n_ttl = len(df_dat_r)

n_smpld = int(n_ttl*n_frac)   
df_ss1 = df_dat_r.sample(n_smpld, replace=False, random_state=42)
# simple_sample = df_pop.sample(n=25, replace=False, random_state=42)
# 표본 갯수 n, 비복원추출(이게 디폴트), 검증위한 시드, 42 for fun. 
df_ss1.head()


,i,gender,ht
15561,15562,1,170.9
13056,13057,2,144.4
5702,5703,1,170.5
3062,3063,2,158.5
9199,9200,2,155.1


In [76]:
# # 2. 연습용 자료, 층화 표본 추출 (그룹별로 1%씩 비율 유지 추출) 
# Better Coding,... 
df_ss2 = (
    df_dat_r.groupby('gender', group_keys=False)
    .sample(frac=n_frac, random_state=42)
)
df_ss2.head()


,i,gender,ht
15862,15863,1,163.1
7185,7186,1,167.5
4804,4805,1,171.5
13485,13486,1,178.6
14926,14927,1,163.5


In [77]:
# df1의 속성들 알아내기.
# 전체 종합 요약	df.info()	행/열/타입/결측치 한눈에 보기
# 행/열 차원 크기	df.shape	(행 개수, 열 개수) 튜플 반환
# 자료 개수 (행)	len(df) 또는 df.shape[0]	데이터 건수
# 변수 개수 (열)	len(df.columns) 또는 df.shape[1]	컬럼 개수
# 변수 이름 목록	df.columns
df_ss1.info()
df_ss2.info()
# df1.shape[0]
# df1.shape[1]
# df1.columns
# print(df_ss1.shape, "\n",  df_ss1.columns) 
print(len(df_dat_r), len(df_ss1), len(df_ss2)) 


<class 'pandas.core.frame.DataFrame'>
Index: 402 entries, 15561 to 8046
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   i       402 non-null    int64  
 1   gender  402 non-null    int64  
 2   ht      402 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 12.6 KB
<class 'pandas.core.frame.DataFrame'>
Index: 403 entries, 15862 to 16411
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   i       403 non-null    int64  
 1   gender  403 non-null    int64  
 2   ht      403 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 12.6 KB
20128 402 403


\begin{align}
\hat\Delta & = \hat \mu_{male} - \hat \mu_{female}  \\
 & \sim N \left(\Delta , SE_{\Delta }^2 \right)  \\
SE^2 & = {\sigma_{male}^2 \over n_{male}} + {\sigma_{female}^2 \over n_{female}}   
\end{align}

In [78]:
# df_ss1(단순추출 자료셋) 또는 df_ss2(층화추롤 자료셋) 을 분석함. 
# df_dat_r 은 그냥 놔두고.
# 이 단계에서 샘플 자료셋을 df_dat으로 명명하고, 분석. 

df_dat = df_ss1    # df_ss1을 다시 명명하고, 분석함. 

gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
hgt = df_dat['ht'].to_numpy()
gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환. 
df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

hgt_m = hgt[gndr == 1]  # 성별 1, m 그룹 키. 그룹별 자료 분리, 배열은 hgt 하나임.
hgt_f = hgt[gndr == 0]  #     0, f. 키 그룹별 자료 분리

n_all = len(df_dat_r)
n_now = len(df_dat) 
n_m = len(hgt_m)
n_f = len(hgt_f)

print(n_all, n_now, n_m, n_f)


20128 402 174 228


### 등분산 검정 결과 참조.
등분산 $ \to $ 상대적으로 간결한 SE 

이분산 $ \to $ 복잡한 자유도 계산 

실제로는 옵션 지정으로 충분.

In [79]:
# 3. 등분산 검정, 정의된 함수 이용. 위 셀.
alpha = 0.05 
phi_hat, pval, ci = variance_ratio_test( hgt_m, hgt_f, alpha) 
print(" 등분산 검정. F ratio, p value :", phi_hat, pval)
print(" 등분산 " if pval > alpha else " 이분산 ")



 등분산 검정. F ratio, p value : 1.0697333066068495 0.6320826127527469
 등분산 


### 그룹(성)별 키의 특성. 평균, 분산(표준편차), 자유도

In [80]:
# 4. 평균, 그룹별 차이 ( m - f ), 표준오차. 
# 추정치: 평균, 분산, 표준오차, overall
# overall, 이 계산은 불필요함. 
muhat = hgt.mean()
var_hgt = hgt.var(ddof=1)  #/  n_all # 이게 통합 분산은 아니지. 

# 그룹별( m - f) 표본평균 및 분산
muhat_m = hgt_m.mean()
muhat_f = hgt_f.mean()

var_hgt_m = hgt_m.var(ddof=1) # /  n_m 
var_hgt_f = hgt_f.var(ddof=1) # /  n_f 


\begin{align}
\hat\Delta & = \hat \mu_{male} - \hat \mu_{female}  \\
 & \sim N \left(\Delta , SE_{\Delta }^2 \right)   
\end{align}

\begin{align}
Var(\hat\Delta) 
   &   = \sigma_{pool}^2 \left( { 1 \over n_{male}} + { 1 \over n_{female}} \right)   \quad \text{등분산인 경우} \\
   &   = {\sigma_{male}^2 \over n_{male}} + {\sigma_{female}^2 \over n_{female}}   \qquad \qquad \text{이분산인 경우} 
\end{align}

In [81]:

# 평균차이
mu_dff = muhat_m - muhat_f 

# 그룹 평균차의 분산, 표준오차 (등분산 vs. 이분산)
#   등분산 가정.
df_pool = n_m + n_f - 2                                    # 등분산 자유도 
sse_pool = (n_m - 1) * var_hgt_m + (n_f - 1) * var_hgt_f   # 등분산 가정 시 적용 가능 
var_pool = sse_pool / df_pool                              # 공통분산 
mu_dff_var_e =  var_pool * ( 1 / n_m + 1 / n_f )            # 평균차의 분산(등분산인 경우). 
se_eq = np.sqrt( mu_dff_var_e )                             # 평균차의 표준오차

#   이분산 가정.
mu_dff_var_u =  var_hgt_m / n_m + var_hgt_f / n_f    # 평균차이 분산, 이분산(일반적) 
se_un = np.sqrt( mu_dff_var_u )                      #         표분오차
df_hetr =  mu_dff_var_u**2 / ( 
        (var_hgt_m/n_m)**2/(n_m-1 )
        + (var_hgt_f/n_f)**2/(n_f-1) )             #   이분산 자유도,  Welch–Satter 방법 

# 평균차이, 표준오차 
print("평균차이 ", mu_dff ) 
print("등분산  (표준오차, 자유도):", se_eq, df_pool ) 
print("이분산  (표준오차, 자유도):", se_un, df_hetr) 


평균차이  13.039322444041119
등분산  (표준오차, 자유도): 0.6189346113462302 400
이분산  (표준오차, 자유도): 0.6217492084100796 365.94544669456934


\begin{align}
T_0 & = { \hat\Delta - \Delta_0 \over SE } \\
  & \sim N(0,1)  \\
  & \sim t_{df}
\end{align}

$ 100(1 - \alpha)\% $ 신뢰구간 
\begin{align}
(X) \quad \hat\Delta - z_{\alpha/2} \times SE < \Delta < \hat\Delta - z_{\alpha/2} \times SE  \\
(O) \quad \hat\Delta - t_{\alpha/2} \times \widehat{ SE } < \Delta < \hat\Delta - t_{\alpha/2} \times \widehat{ SE } 
\end{align}


In [82]:

# 신뢰구간 찾기, right, left, 등분산, 이분산. 
# 임계치 
t_2_eq = stats.t.ppf( 1- alpha/2, df_pool )  # critical value on the right, two side
t_2_un = stats.t.ppf( 1- alpha/2, df_hetr )  # critical value on the right, two side
#    zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side

# 신뢰구간(등분산)
ci_r = mu_dff + t_2_eq * se_eq 
ci_l = mu_dff - t_2_eq * se_eq   

print( "등분산 가정:")
print( "      mean diff,    se of mu_diff,     confidence interval ")
print( mu_dff, se_eq, ci_l, ci_r ) 

# 신뢰구간(이분산) --- 임계치와 표준오차 조정/변경/선택, 자유도...  
ci_r = mu_dff + t_2_un * se_un 
ci_l = mu_dff - t_2_un * se_un   

print( "이분산 가정:")
print( "      mean diff,    se of mu_diff,     confidence interval ")
print( mu_dff, se_un, ci_l, ci_r ) 


등분산 가정:
      mean diff,    se of mu_diff,     confidence interval 
13.039322444041119 0.6189346113462302 11.822551251943679 14.256093636138559
이분산 가정:
      mean diff,    se of mu_diff,     confidence interval 
13.039322444041119 0.6217492084100796 11.816672711288387 14.26197217679385


\begin{align}
\text{ reject  H0  if } \quad |T_0| > t_2 
\end{align}
where 
$ P( T_0 > t_2 ) = P(T_0 < t_1 ) = \alpha/2 $

In [83]:
# 가설검정. 전체, 톨 
# 귀무가설 H0: mu = mu0
# 검정통계치, t_0, pvalue 
mu_dff_zero = 0
print("H0: 두 그룹 평균이 동일하다")
print(" H0 : mean difference =", mu_dff_zero )

t_0eq = np.abs( ( mu_dff - mu_dff_zero ) / se_eq  )       # 등분산 
t_0un = np.abs( ( mu_dff - mu_dff_zero ) / se_un  )       # 이분산

yn_h0 = " 'reject h0' " if t_0eq > t_2_eq else " 'fail to reject h0' "   # 이건 되는 군. 
pval = 2* stats.t.sf( t_0eq, df_pool ) 

print(" 등분산 가정 t-검정  ") 
print(" t0, t_(a/2), 판단, pvalue : ", t_0eq , t_2_eq, yn_h0, pval) 

yn_h0 = " 'reject h0' " if t_0eq > t_2_un else " 'fail to reject h0' "   # 이건 되는 군. 
pval = 2* stats.t.sf( t_0un, df_hetr ) 

print(" 이분산 가정 t-검정  ") 
print(" t0, t_(a/2), 판단, pvalue : ", t_0un , t_2_un, yn_h0, pval) 

print("등분산; ")
print(" 분산, 자유도(등) ")
print( mu_dff_var_e, df_pool )
print("등분산; ")
print(" 분산, 자유도(이) ")
print( mu_dff_var_u, df_hetr)


H0: 두 그룹 평균이 동일하다
 H0 : mean difference = 0
 등분산 가정 t-검정  
 t0, t_(a/2), 판단, pvalue :  21.067366737949254 1.965912343229391  'reject h0'  7.939422008002053e-67
 이분산 가정 t-검정  
 t0, t_(a/2), 판단, pvalue :  20.971996856071474 1.966467695036169  'reject h0'  1.0673979956328071e-64
등분산; 
 분산, 자유도(등) 
0.38308005312230897 400
등분산; 
 분산, 자유도(이) 
0.38657207815856065 365.94544669456934


### provided by built-in modules

In [84]:
# 모듈 이용 scipy.stats.ttest_ind 
# 두 그룹 평균 비교.
# 이 모든 것을 간단 코드로 실현.
t_0_homo, pval_homo = ttest_ind(hgt_f, hgt_m, equal_var=True)  # 등분산 가정 평균 비교
print("equal variance, t0, pvalue ")
print(t_0_homo, pval_homo)

t_0_hetr, pval_hetr = ttest_ind(hgt_f, hgt_m, equal_var=False) # 이분산 가정 평균 비교 
print("not equal variance, t0, pvalue ")
print(t_0_hetr, pval_hetr)


equal variance, t0, pvalue 
-21.067366737949254 7.939422008002053e-67
not equal variance, t0, pvalue 
-20.971996856071474 1.0673979956328075e-64
